In [1]:
# Hugging Face token setup
# Recommended: add a Kaggle Secret named HF_TOKEN and enable it for this notebook.
# If you prefer a variable cell, paste a NEW read token below.
hf_token = 'your_hugging_face_token_here'

import os

try:
    hf_token
except NameError:
    hf_token = os.environ.get("HF_TOKEN", "")

if hf_token:
    os.environ["HF_TOKEN"] = str(hf_token).strip()

# Default is dynamic balancing (100 core / 50 extra). Change TTS_PER_LANGUAGE to a number (e.g. '25') to override.
os.environ["TTS_PER_LANGUAGE"] = os.environ.get("TTS_PER_LANGUAGE", "dynamic")
os.environ["MAKE_ZIP"] = os.environ.get("MAKE_ZIP", "1")
os.environ["USE_MULTI_GPU_DEVICE_MAP"] = os.environ.get("USE_MULTI_GPU_DEVICE_MAP", "1")
os.environ["OUT_DIR"] = os.environ.get("OUT_DIR", "/kaggle/working/indictts_tts_only_dataset")

In [2]:
from __future__ import annotations

import csv
import gc
import hashlib
import importlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

MAIN_DATASET = "SherryT997/IndicTTS-Deepfake-Challenge-Data"
SVARA_MODEL_ID = "kenpath/svara-tts-v1"
TARGET_SR = 16000
SEED = int(os.getenv("SEED", "42"))
TTS_PER_LANGUAGE = os.getenv("TTS_PER_LANGUAGE", "dynamic")
MAX_PROMPT_SCAN = int(os.getenv("MAX_PROMPT_SCAN", "30000"))
TTS_PROGRESS_EVERY = int(os.getenv("TTS_PROGRESS_EVERY", "2"))
MAKE_ZIP = os.getenv("MAKE_ZIP", "1").strip().lower() not in {"0", "false", "no"}
USE_MULTI_GPU_DEVICE_MAP = os.getenv("USE_MULTI_GPU_DEVICE_MAP", "1").strip().lower() not in {"0", "false", "no"}
STRICT_FAILURES = os.getenv("STRICT_FAILURES", "0").strip().lower() in {"1", "true", "yes"}

OUT_DIR = Path(os.getenv("OUT_DIR", "/kaggle/working/indictts_tts_only_dataset"))
AUDIO_DIR = OUT_DIR / "audio"
METADATA_PATH = OUT_DIR / "metadata.csv"
LOG_PATH = OUT_DIR / "run_log.txt"
STATUS_PATH = OUT_DIR / "run_status.json"
SVARA_REPO_DIR = Path(os.getenv("SVARA_REPO_DIR", "/kaggle/working/svara_repo"))

DEFAULT_LANGUAGES = [
    "Assamese", "Bengali", "Bodo", "Dogri", "Kannada", "Malayalam",
    "Marathi", "Sanskrit", "Nepali", "English", "Telugu", "Hindi",
    "Odia", "Manipuri", "Gujarati", "Tamil",
]
LANGUAGES = [x.strip() for x in os.getenv("LANGUAGES", ",".join(DEFAULT_LANGUAGES)).split(",") if x.strip()]

def get_target_limit(language: str) -> int:
    # If the user has set a numeric override for TTS_PER_LANGUAGE, use it
    val = str(TTS_PER_LANGUAGE).strip()
    if val.isdigit():
        return int(val)
    # Otherwise, use the dynamic balancing limits
    # 10 languages with 100 real human speech clips target 100 synthetic clips
    if language in {"Assamese", "Bengali", "English", "Gujarati", "Hindi", "Kannada", "Malayalam", "Marathi", "Tamil", "Telugu"}:
        return 100
    # 4 languages with 0 real human speech clips target 50 synthetic clips
    if language in {"Bodo", "Dogri", "Nepali", "Sanskrit"}:
        return 50
    return 0

SVARA_SPEAKER_NAME = {
    "Hindi": "Hindi",
    "Bengali": "Bengali",
    "Marathi": "Marathi",
    "Telugu": "Telugu",
    "Kannada": "Kannada",
    "Assamese": "Assamese",
    "Bodo": "Bodo",
    "Dogri": "Dogri",
    "Gujarati": "Gujarati",
    "Malayalam": "Malayalam",
    "Tamil": "Tamil",
    "Nepali": "Nepali",
    "Sanskrit": "Sanskrit",
    "English": "English (Indian)",
}
SVARA_EMOTIONS = ["<clear>", "<happy>", "<sad>", "<fear>", "<anger>"]
GENDERS = ["Male", "Female"]

# Used only if the challenge dataset cannot provide enough prompts for a language.
FALLBACK_PROMPTS = {
    "English": ["This is a clear speech sample for the dataset.", "The speaker is reading a short sentence."],
    "Hindi": ["यह एक स्पष्ट आवाज़ का नमूना है।", "आज मौसम बहुत अच्छा है।"],
    "Bengali": ["এটি একটি পরিষ্কার কণ্ঠের নমুনা।", "আজ আবহাওয়া খুব ভালো।"],
    "Tamil": ["இது தெளிவான குரல் மாதிரி ஆகும்.", "இன்று வானிலை மிகவும் நன்றாக உள்ளது."],
    "Telugu": ["ఇది స్పష్టమైన స్వర నమూనా.", "ఈ రోజు వాతావరణం చాలా బాగుంది."],
    "Kannada": ["ಇದು ಸ್ಪಷ್ಟವಾದ ಧ್ವನಿ ಮಾದರಿ.", "ಇಂದು ಹವಾಮಾನ ತುಂಬಾ ಚೆನ್ನಾಗಿದೆ."],
    "Malayalam": ["ഇത് വ്യക്തമായ ശബ്ദ സാമ്പിളാണ്.", "ഇന്ന് കാലാവസ്ഥ വളരെ നല്ലതാണ്."],
    "Marathi": ["हा स्पष्ट आवाजाचा नमुना आहे.", "आज हवामान खूप चांगले आहे."],
    "Gujarati": ["આ સ્પષ્ટ અવાજનો નમૂનો છે.", "આજે હવામાન ખૂબ સારું છે."],
    "Odia": ["ଏହା ଏକ ସ୍ପଷ୍ଟ କଣ୍ଠସ୍ୱର ନମୁନା।", "ଆଜି ପାଗ ବହୁତ ଭଲ ଅଛି।"],
    "Assamese": ["এইটো এটা স্পষ্ট কণ্ঠৰ নমুনা।", "আজিৰ বতৰ অতি ভাল।"],
    "Nepali": ["यो स्पष्ट आवाजको नमुना हो।", "आज मौसम धेरै राम्रो छ।"],
    "Sanskrit": ["एषः स्पष्टस्वरस्य नमूना अस्ति।", "अद्य वातावरणं सुन्दरम् अस्ति।"],
    "Bodo": ["बेयो मोनसे रोखा सोदोबनि नमुना।"],
    "Dogri": ["एह साफ आवाज दा नमूना ऐ।"],
    "Manipuri": ["মসি শেংবা খোঞ্জেলগী নমুনা অমনি।"],
}


def log(message: str) -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {message}"
    print(line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")


def write_status(status: str, extra: Optional[dict] = None) -> None:
    payload = {
        "status": status,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "out_dir": str(OUT_DIR),
        "metadata_path": str(METADATA_PATH),
        "tts_per_language": TTS_PER_LANGUAGE,
        "languages": LANGUAGES,
    }
    if extra:
        payload.update(extra)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    STATUS_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def run_cmd(args: Sequence[str], description: str, required: bool = True) -> bool:
    log(f"{description}: {' '.join(args)}")
    try:
        subprocess.run(list(args), check=True)
        return True
    except Exception as exc:
        log(f"FAILED: {description}: {exc}")
        if required:
            raise
        return False


def import_available(module_name: str) -> bool:
    try:
        importlib.import_module(module_name)
        return True
    except Exception:
        return False


def pip_install(packages: Sequence[str], required: bool = True) -> bool:
    return run_cmd([sys.executable, "-m", "pip", "install", "-q", *packages], f"Installing {', '.join(packages)}", required)


def ensure_dependencies() -> None:
    checks = [
        ("datasets", "datasets[audio]"),
        ("soundfile", "soundfile"),
        ("librosa", "librosa"),
        ("numpy", "numpy"),
        ("tqdm", "tqdm"),
        ("huggingface_hub", "huggingface_hub"),
        ("transformers", "transformers"),
        ("accelerate", "accelerate"),
        ("snac", "snac"),
        ("langcodes", "langcodes"),
    ]
    missing = [pkg for mod, pkg in checks if not import_available(mod)]
    if missing:
        pip_install(missing, required=True)

    expected_svara_file = SVARA_REPO_DIR / "tts_engine" / "encoder.py"
    if not expected_svara_file.exists():
        if SVARA_REPO_DIR.exists():
            log(f"Svara repo exists but looks incomplete: {SVARA_REPO_DIR}")
        else:
            run_cmd(["git", "clone", "--depth", "1", "https://github.com/Kenpath/svara-tts-inference.git", str(SVARA_REPO_DIR)], "Cloning Svara inference repo", required=True)

    still_missing = [mod for mod, _ in checks if not import_available(mod)]
    if still_missing:
        raise RuntimeError(f"Dependency import check failed: {still_missing}")
    if not expected_svara_file.exists():
        raise RuntimeError(f"Svara inference repo missing expected file: {expected_svara_file}")
    log("Dependencies are available.")


def clean_token(value: object) -> Optional[str]:
    if value is None:
        return None
    token = str(value).strip().strip('"').strip("'")
    if not token or token.lower() in {"none", "null"}:
        return None
    return token


def get_hf_token() -> Optional[str]:
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_TOKEN", "hf_token"):
        token = clean_token(os.environ.get(name))
        if token:
            os.environ["HF_TOKEN"] = token
            os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", token)
            log(f"Using Hugging Face token from {name} environment variable.")
            return token
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_TOKEN"):
            try:
                token = clean_token(secrets.get_secret(name))
            except Exception:
                continue
            if token:
                os.environ["HF_TOKEN"] = token
                log(f"Using Hugging Face token from Kaggle secret {name}.")
                return token
    except Exception as exc:
        log(f"Kaggle secrets unavailable or not enabled: {exc}")
    try:
        from huggingface_hub import get_token
        token = clean_token(get_token())
        if token:
            os.environ["HF_TOKEN"] = token
            log("Using Hugging Face token from cached huggingface_hub login.")
            return token
    except Exception as exc:
        log(f"Could not read cached Hugging Face token: {exc}")
    log("No Hugging Face token found. Public datasets/models will still be tried.")
    return None


def log_torch_runtime() -> None:
    import torch
    if not torch.cuda.is_available():
        log("Torch runtime: CUDA is not available; TTS will be slow on CPU.")
        return
    devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    log(f"Torch runtime: CUDA devices={devices}, device_count={torch.cuda.device_count()}, multi_gpu_device_map={USE_MULTI_GPU_DEVICE_MAP}")


def text_hash(text: str) -> str:
    return hashlib.md5("".join(str(text).strip().lower().split()).encode("utf-8")).hexdigest()


def slug(text: str) -> str:
    return "".join(ch.lower() if ch.isalnum() else "-" for ch in text).strip("-")


def read_metadata() -> List[dict]:
    if not METADATA_PATH.exists():
        return []
    with METADATA_PATH.open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_metadata(rows: List[dict]) -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    fields = ["file_name", "id", "text", "language", "is_tts", "source"]
    with METADATA_PATH.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fields})


def count_tts_by_language(rows: Sequence[dict]) -> Dict[str, int]:
    counts = {language: 0 for language in LANGUAGES}
    for row in rows:
        if str(row.get("is_tts")) == "1" and row.get("language") in counts:
            counts[row["language"]] += 1
    return counts


def build_existing_keys(rows: Sequence[dict]) -> set[str]:
    return {text_hash(f"{row.get('language', '')}::{row.get('text', '')}") for row in rows if row.get("text")}


def collect_prompts() -> Dict[str, List[str]]:
    from datasets import load_dataset

    prompts: Dict[str, List[str]] = {language: [] for language in LANGUAGES}
    used: set[str] = set()
    missing = {lang for lang in LANGUAGES if get_target_limit(lang) > 0}
    log(f"Collecting TTS prompts from challenge train split for {len(missing)} languages.")
    try:
        ds = load_dataset(MAIN_DATASET, split="train", streaming=True, token=HF_TOKEN)
        ds = ds.shuffle(seed=SEED, buffer_size=5000)
        scanned = 0
        for row in ds:
            scanned += 1
            if scanned == 1 or scanned % 1000 == 0:
                log(f"Prompt scan: scanned={scanned}, missing={sorted(missing)}")
            language = row.get("language")
            if language not in missing:
                continue
            if row.get("is_tts") not in (0, "0", False):
                continue
            text = row.get("text")
            if not text:
                continue
            key = text_hash(f"{language}::{text}")
            if key in used:
                continue
            prompts[language].append(str(text))
            used.add(key)
            limit = get_target_limit(language)
            if len(prompts[language]) >= limit:
                missing.remove(language)
                log(f"{language}: collected {len(prompts[language])} prompts")
                if not missing:
                    break
            if scanned >= MAX_PROMPT_SCAN:
                log(f"Stopped prompt scan at MAX_PROMPT_SCAN={MAX_PROMPT_SCAN}")
                break
    except Exception as exc:
        log(f"Could not collect prompts from challenge dataset: {exc}")

    for language in LANGUAGES:
        fallback = FALLBACK_PROMPTS.get(language, [])
        i = 0
        limit = get_target_limit(language)
        while len(prompts[language]) < limit and fallback:
            base = fallback[i % len(fallback)]
            text = base if i < len(fallback) else f"{base} {i + 1}."
            prompts[language].append(text)
            i += 1
        log(f"{language}: prompts ready={len(prompts[language])}")
    return prompts


def torch_device_settings(torch) -> Tuple[str, object, bool]:
    if not torch.cuda.is_available():
        return "cpu", torch.float32, False
    device = "cuda:0"
    dtype = torch.float16
    use_device_map = USE_MULTI_GPU_DEVICE_MAP and torch.cuda.device_count() > 1
    return device, dtype, use_device_map


def setup_svara():
    import torch
    from transformers import AutoModelForCausalLM

    os.environ.setdefault("SNAC_COMPILE", "false")
    if str(SVARA_REPO_DIR) not in sys.path:
        sys.path.insert(0, str(SVARA_REPO_DIR))
    from tts_engine.codec import SNACCodec, get_or_load_tokenizer
    from tts_engine.constants import AUDIO_TOKEN_OFFSETS, END_OF_SPEECH
    from tts_engine.encoder import svara_text_to_tokens

    device, dtype, use_device_map = torch_device_settings(torch)
    kwargs = {"torch_dtype": dtype, "low_cpu_mem_usage": True}
    if use_device_map:
        kwargs["device_map"] = "auto"
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    tokenizer = get_or_load_tokenizer(SVARA_MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(SVARA_MODEL_ID, **kwargs)
    if not use_device_map:
        model = model.to(device)
    model = model.eval()
    log(f"Loaded Svara on device={device}, dtype={dtype}, device_map={getattr(model, 'hf_device_map', None)}")
    codec = SNACCodec(device=device)
    return {
        "device": device,
        "model": model,
        "tokenizer": tokenizer,
        "codec": codec,
        "text_to_tokens": svara_text_to_tokens,
        "offsets": AUDIO_TOKEN_OFFSETS,
        "eos": END_OF_SPEECH,
    }


def svara_decode_full(codes: List[int], ctx) -> Optional[object]:
    import torch

    frames = len(codes) // 7
    if frames == 0:
        return None
    frame = codes[: frames * 7]
    t = torch.tensor(frame, dtype=torch.int32, device=ctx["device"]).view(frames, 7)
    codes_0 = t[:, 0].reshape(1, -1)
    codes_1 = t[:, [1, 4]].reshape(1, -1)
    codes_2 = t[:, [2, 3, 5, 6]].reshape(1, -1)
    with torch.inference_mode():
        audio = ctx["codec"].model.decode([codes_0, codes_1, codes_2])
    return audio.detach().float().cpu().numpy().reshape(-1)


def generate_svara(text: str, language: str, index: int, ctx) -> Optional[Tuple[object, int]]:
    import torch

    speaker_base = SVARA_SPEAKER_NAME.get(language)
    if not speaker_base:
        return None
    speaker_id = f"{speaker_base} ({GENDERS[index % len(GENDERS)]})"
    emotion = SVARA_EMOTIONS[index % len(SVARA_EMOTIONS)]
    prompt_ids = ctx["text_to_tokens"](
        text=f"{text} {emotion}",
        speaker_id=speaker_id,
        tokenizer=ctx["tokenizer"],
        return_decoded=False,
    )
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=ctx["device"])
    with torch.inference_mode():
        out = ctx["model"].generate(
            input_ids=input_ids,
            max_new_tokens=1536,
            do_sample=True,
            temperature=0.75,
            top_p=0.9,
            top_k=40,
            repetition_penalty=1.1,
            eos_token_id=ctx["eos"],
            pad_token_id=ctx["tokenizer"].eos_token_id,
        )
    generated_ids = out[0][input_ids.shape[1] :].tolist()
    codes, good = [], 0
    for token_id in generated_ids:
        if token_id == ctx["eos"]:
            break
        band = good % 7
        code = token_id - ctx["offsets"][band]
        if 0 <= code < 4096:
            codes.append(code)
            good += 1
    if len(codes) < 7:
        return None
    arr = svara_decode_full(codes, ctx)
    if arr is None:
        return None
    return arr, int(ctx["codec"].sample_rate)


def valid_audio(arr, sr: int, min_s: float = 0.25, max_s: float = 25.0) -> bool:
    import numpy as np
    if arr is None or sr is None:
        return False
    arr = np.asarray(arr).reshape(-1)
    duration = len(arr) / float(sr)
    return bool(min_s <= duration <= max_s and np.std(arr) > 1e-4)


def resample_to_target(arr, sr: int):
    import librosa
    import numpy as np
    arr = np.asarray(arr, dtype=np.float32).reshape(-1)
    if sr == TARGET_SR:
        return arr
    return librosa.resample(arr, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)


def write_audio_clip(clip_id: str, arr, sr: int) -> str:
    import soundfile as sf
    AUDIO_DIR.mkdir(parents=True, exist_ok=True)
    arr = resample_to_target(arr, sr)
    rel_path = f"audio/{clip_id}.wav"
    sf.write(OUT_DIR / rel_path, arr, TARGET_SR, subtype="PCM_16")
    return rel_path


def build_tts_dataset() -> List[dict]:
    rows = read_metadata()
    if rows:
        log(f"Resuming existing TTS metadata rows: {len(rows)}")
    prompts = collect_prompts()
    existing = build_existing_keys(rows)
    counts = count_tts_by_language(rows)
    ctx = setup_svara()

    for language in LANGUAGES:
        if language not in SVARA_SPEAKER_NAME:
            log(f"{language}: no Svara speaker mapping; skipping")
            continue
        limit = get_target_limit(language)
        have = counts.get(language, 0)
        need = max(0, limit - have)
        if need == 0:
            log(f"{language}: already has {have} TTS clips (target={limit})")
            continue
        accepted = 0
        attempted = 0
        started = time.time()
        log(f"{language}: generating {need} TTS clips")
        for text in prompts.get(language, []):
            if accepted >= need:
                break
            attempted += 1
            if attempted == 1 or attempted % TTS_PROGRESS_EVERY == 0:
                log(f"{language}: progress accepted={accepted}/{need}, attempted={attempted}, elapsed={int(time.time() - started)}s")
            key = text_hash(f"{language}::{text}")
            if key in existing:
                continue
            try:
                result = generate_svara(text, language, have + accepted, ctx)
                if result is None:
                    log(f"{language}: generation returned no audio for attempt={attempted}")
                    continue
                arr, sr = result
                if not valid_audio(arr, sr):
                    log(f"{language}: generated audio failed validation for attempt={attempted}")
                    continue
                clip_id = f"tts-{slug(language)}-{text_hash(text)[:12]}-{have + accepted:04d}"
                rel_path = write_audio_clip(clip_id, arr, sr)
                rows.append({
                    "file_name": rel_path,
                    "id": clip_id,
                    "text": str(text),
                    "language": language,
                    "is_tts": "1",
                    "source": SVARA_MODEL_ID,
                })
                existing.add(key)
                accepted += 1
                write_metadata(rows)
                if accepted == need or accepted % TTS_PROGRESS_EVERY == 0:
                    log(f"{language}: accepted progress {accepted}/{need} after {int(time.time() - started)}s")
            except RuntimeError as exc:
                log(f"{language}: runtime error on attempt={attempted}: {exc}")
                if "out of memory" in str(exc).lower() or "cuda" in str(exc).lower():
                    import torch
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                continue
            except Exception as exc:
                log(f"{language}: failed attempt={attempted}: {exc}")
                continue
        log(f"{language}: TTS accepted {accepted}/{need}")
    return rows


def write_readme(rows: Sequence[dict]) -> None:
    counts = count_tts_by_language(rows)
    lines = [
        "# IndicTTS TTS-Only Dataset",
        "",
        "Audiofolder-style dataset containing generated TTS clips only.",
        "",
        "Columns: `file_name`, `id`, `text`, `language`, `is_tts`, `source`.",
        "",
        "| language | tts |",
        "|---|---:|",
    ]
    for language in LANGUAGES:
        lines.append(f"| {language} | {counts.get(language, 0)} |")
    (OUT_DIR / "README.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def make_zip() -> Optional[Path]:
    if not MAKE_ZIP:
        return None
    archive = shutil.make_archive(str(OUT_DIR.with_suffix("")), "zip", root_dir=OUT_DIR)
    log(f"Created zip: {archive}")
    return Path(archive)


def main() -> int:
    random.seed(SEED)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    AUDIO_DIR.mkdir(parents=True, exist_ok=True)
    write_status("running")
    try:
        log("Starting IndicTTS TTS-only dataset generation.")
        log(
            "Settings: "
            f"TTS_PER_LANGUAGE={TTS_PER_LANGUAGE}, "
            f"MAX_PROMPT_SCAN={MAX_PROMPT_SCAN}, "
            f"MAKE_ZIP={MAKE_ZIP}, "
            f"USE_MULTI_GPU_DEVICE_MAP={USE_MULTI_GPU_DEVICE_MAP}, "
            f"languages={LANGUAGES}"
        )
        ensure_dependencies()
        log_torch_runtime()
        global HF_TOKEN
        HF_TOKEN = get_hf_token()
        rows = build_tts_dataset()
        write_metadata(rows)
        write_readme(rows)
        zip_path = make_zip()
        counts = count_tts_by_language(rows)
        write_status("complete", {"row_count": len(rows), "counts": counts, "zip_path": str(zip_path) if zip_path else None})
        log(f"Complete. Total TTS rows: {len(rows)}")
        return 0
    except Exception as exc:
        log(f"FATAL: {exc}")
        log(traceback.format_exc())
        write_status("failed", {"error": str(exc), "traceback": traceback.format_exc()})
        if STRICT_FAILURES:
            raise
        return 0


if __name__ == "__main__":
    raise SystemExit(main())

[2026-07-02 02:39:35] Starting IndicTTS TTS-only dataset generation.
[2026-07-02 02:39:35] Settings: TTS_PER_LANGUAGE=10, MAX_PROMPT_SCAN=30000, MAKE_ZIP=True, USE_MULTI_GPU_DEVICE_MAP=True, languages=['Assamese', 'Bengali', 'Bodo', 'Dogri', 'Kannada', 'Malayalam', 'Marathi', 'Sanskrit', 'Nepali', 'English', 'Telugu', 'Hindi', 'Odia', 'Manipuri', 'Gujarati', 'Tamil']
[2026-07-02 02:40:14] Installing snac, langcodes: /usr/bin/python3 -m pip install -q snac langcodes
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 4.0 MB/s eta 0:00:00
[2026-07-02 02:40:19] Cloning Svara inference repo: git clone --depth 1 https://github.com/Kenpath/svara-tts-inference.git /kaggle/working/svara_repo


Cloning into '/kaggle/working/svara_repo'...


[2026-07-02 02:40:20] Dependencies are available.
[2026-07-02 02:40:20] Torch runtime: CUDA devices=['Tesla T4', 'Tesla T4'], device_count=2, multi_gpu_device_map=True
[2026-07-02 02:40:20] Using Hugging Face token from HF_TOKEN environment variable.
[2026-07-02 02:40:20] Collecting TTS prompts from challenge train split for 16 languages.


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/79.5M [00:00<?, ?B/s]

[2026-07-02 02:47:38] Assamese: generating 10 TTS clips
[2026-07-02 02:47:38] Assamese: progress accepted=0/10, attempted=1, elapsed=0s
[2026-07-02 02:48:23] Assamese: progress accepted=1/10, attempted=2, elapsed=45s
[2026-07-02 02:48:40] Assamese: accepted progress 2/10 after 61s
[2026-07-02 02:49:05] Assamese: progress accepted=3/10, attempted=4, elapsed=87s
[2026-07-02 02:49:32] Assamese: accepted progress 4/10 after 113s
[2026-07-02 02:50:05] Assamese: progress accepted=5/10, attempted=6, elapsed=146s
[2026-07-02 02:50:39] Assamese: accepted progress 6/10 after 180s
[2026-07-02 02:51:19] Assamese: progress accepted=7/10, attempted=8, elapsed=220s
[2026-07-02 02:51:57] Assamese: accepted progress 8/10 after 258s
[2026-07-02 02:52:42] Assamese: progress accepted=9/10, attempted=10, elapsed=303s
[2026-07-02 02:52:58] Assamese: accepted progress 10/10 after 320s
[2026-07-02 02:52:58] Assamese: TTS accepted 10/10
[2026-07-02 02:52:58] Bengali: generating 10 TTS clips
[2026-07-02 02:52:5

SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
